In [ ]:
"""
============================================================
  Aviation Stock Performance & Oil Price Analysis
  Data Collection Script
============================================================
"""

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
import time

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 1. CONFIGURATION – Tickers & Date Range
# ─────────────────────────────────────────────

START_DATE = "2015-01-01"
END_DATE   = "2026-02-21"  

AIRLINE_TICKERS = {
    "Qatar Airways"               : None,          # private
    "Singapore Airlines"          : "C6L.SI",
    "Cathay Pacific Airways"      : "0293.HK",
    "Emirates"                    : None,          # private
    "ANA All Nippon Airways"      : "9202.T",
    "Turkish Airlines"            : "THYAO.IS",
    "Korean Air"                  : "003490.KS",
    "Air France-KLM"              : "AF.PA",
    "Japan Airlines"              : "9201.T",
    "Hainan Airlines"             : "600221.SS",
    "Swiss International Air Lines": None,         # subsidiary of Lufthansa, no separate ticker
    "EVA Air"                     : "2618.TW",
    "British Airways (IAG)"       : "IAG.L",
    "Qantas Airways"              : "QAN.AX",
    "Lufthansa"                   : "LHA.DE",
    "Virgin Atlantic"             : None,          # private
    "Saudia"                      : None,          # private
    "STARLUX Airlines"            : None,          # private (Taiwan)
    "Air Canada"                  : "AC.TO",
    "Iberia (IAG)"                : "IAG.MC",
    "Delta Air Lines"             : "DAL",
    "Austrian Airlines"           : None,          # subsidiary of Lufthansa
    "Air New Zealand"             : "AIR.NZ",
    "Finnair"                     : "FIA1S.HE",
    "Etihad Airways"              : None,          # private
    "Malaysia Airlines"           : None,          # delisted (private since 2014)
    "AirAsia"                     : "AIRASIA.KL",
    "Thai Airways"                : "THAI.BK",
    "Scoot"                       : None,          # subsidiary of Singapore Airlines
    "Fiji Airways"                : None,          # private
    "Bangkok Airways"             : "BA.BK",
    "China Southern Airlines"     : "1055.HK",
    "Virgin Australia"            : None,          # delisted 2020
    "Gulf Air"                    : None,          # private
    "Air Astana"                  : "AIRA.L",
    "China Airlines"              : "2610.TW",
    "Ethiopian Airlines"          : None,          # private
    "IndiGo"                      : "INDIGO.NS",
    "Eurowings"                   : None,          # subsidiary of Lufthansa
    "Asiana Airlines"             : "020560.KS",
    "Vueling Airlines"            : None,          # subsidiary of IAG
    "LATAM Airlines"              : "LTM",
    "Porter Airlines"             : None,          # private
    "Volotea"                     : None,          # private
    "Garuda Indonesia"            : "GIAA.JK",
    "Aegean Airlines"             : "AEGN.AT",
    "Oman Air"                    : None,          # private
    "Transavia France"            : None,          # subsidiary of Air France-KLM
    "Azerbaijan Airlines"         : None,          # private
    "Gulfstream (General Dynamics)": "GD",         # Gulfstream is a GD division; nearest public proxy
}

# Oil futures tickers
OIL_TICKERS = {
    "Brent Crude" : "BZ=F",
    "WTI Crude"   : "CL=F",
}


# ─────────────────────────────────────────────
# 2. HELPER FUNCTIONS
# ─────────────────────────────────────────────

def download_ticker_data(ticker_symbol: str, start: str, end: str) -> pd.DataFrame | None:
    """
    Download daily OHLCV data for a single ticker from Yahoo Finance.

    Parameters
    ----------
    ticker_symbol : str  – Yahoo Finance ticker symbol
    start         : str  – Start date  (YYYY-MM-DD)
    end           : str  – End date    (YYYY-MM-DD)

    Returns
    -------
    pd.DataFrame or None
        Raw DataFrame with Date + OHLCV columns, or None if no data.
    """
    try:
        df = yf.download(
            tickers    = ticker_symbol,
            start      = start,
            end        = end,
            auto_adjust= False,     # keep 'Adj Close' column explicitly
            progress   = False,     # suppress download bar
            threads    = False,     # single-threaded for stability
        )

        if df.empty:
            return None

        return df

    except Exception as exc:
        return None


def clean_and_enrich(df: pd.DataFrame, ticker_symbol: str) -> pd.DataFrame:
    """
    Clean the raw DataFrame and add feature-engineering columns.

    Steps
    -----
    1. Flatten MultiIndex columns produced by yfinance.
    2. Reset index (Date becomes a regular column).
    3. Rename columns to lowercase.
    4. Drop rows where 'close' is NaN.
    5. Add 'daily_return'       – percentage change of Close price.
    6. Add 'rolling_volatility' – 30-day rolling std of daily_return.
    7. Add 'ticker' column.
    8. Forward-fill then back-fill remaining NaN values.

    Parameters
    ----------
    df            : pd.DataFrame – Raw DataFrame from yfinance
    ticker_symbol : str          – Ticker label to add as column

    Returns
    -------
    pd.DataFrame – Cleaned & enriched DataFrame
    """

    if isinstance(df.columns, pd.MultiIndex):
        # Keep only the first level (Price) to drop the ticker sub-level
        df.columns = [col[0] for col in df.columns]

    # --- Reset index so 'Date' becomes a regular column ---
    df = df.reset_index()

    # --- Lowercase all column names & strip whitespace ---
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

    # --- Rename 'adj_close' if it appears as 'adj close' ---
    df.rename(columns={"adj close": "adj_close"}, inplace=True)

    # --- Drop rows with missing Close price (core metric) ---
    df.dropna(subset=["close"], inplace=True)

    # --- Add daily return (percentage change) ---
    # pct_change() computes (current - previous) / previous * 100
    df["daily_return"] = df["close"].pct_change() * 100

    # --- Add 30-day rolling volatility (std of daily_return) ---
    df["rolling_volatility_30d"] = (
        df["daily_return"]
          .rolling(window=30, min_periods=10)   # min 10 obs to avoid NaN blocks
          .std()
    )

    # --- Add ticker identifier column ---
    df["ticker"] = ticker_symbol

    # --- Handle remaining missing values ---
    # Forward-fill first (carry last known value forward)
    df.ffill(inplace=True)
    # Back-fill remaining NaN at the start of series
    df.bfill(inplace=True)

    # --- Reorder columns: date, ticker, then the rest ---
    priority_cols = ["date", "ticker"]
    other_cols    = [c for c in df.columns if c not in priority_cols]
    df = df[priority_cols + other_cols]

    # --- Reset index cleanly (drop old numeric index) ---
    df.reset_index(drop=True, inplace=True)

    return df


def build_airlines_dataframe(
    ticker_map: dict,
    start: str,
    end: str,
    delay_secs: float = 0.5
) -> pd.DataFrame:
    """
    Iterate over every airline ticker, download & clean data,
    and concatenate all results into a single DataFrame.

    Parameters
    ----------
    ticker_map  : dict  – {airline_name: yahoo_ticker or None}
    start       : str   – Start date
    end         : str   – End date
    delay_secs  : float – Pause between API calls (rate-limit courtesy)

    Returns
    -------
    pd.DataFrame – Combined airline stock DataFrame
    """
    all_frames = []
    total      = len(ticker_map)

    for idx, (airline_name, ticker_symbol) in enumerate(ticker_map.items(), start=1):
        print(f"  [{idx:02d}/{total}] {airline_name}", end="")

        # Skip privately-held companies that have no public ticker
        if ticker_symbol is None:
            print(f" → SKIPPED (privately held, no public ticker)")
            continue

        print(f" ({ticker_symbol}) … ", end="", flush=True)

        raw_df = download_ticker_data(ticker_symbol, start, end)

        if raw_df is None:
            print("NO DATA")
            continue

        clean_df = clean_and_enrich(raw_df, ticker_symbol)

        # Also store the human-readable airline name for reference
        clean_df.insert(1, "airline_name", airline_name)

        all_frames.append(clean_df)
        print(f"OK  ({len(clean_df):,} rows)")

        # Small pause to be polite to Yahoo Finance's servers
        time.sleep(delay_secs)

    if not all_frames:
        raise ValueError("No airline data was downloaded. Check tickers or internet connection.")

    combined = pd.concat(all_frames, ignore_index=True)
    print(f"\n  Total airline rows combined: {len(combined):,}")
    return combined


def build_oil_dataframe(
    oil_map: dict,
    start: str,
    end: str
) -> pd.DataFrame:
    """
    Download and clean oil price data for all oil tickers.

    Parameters
    ----------
    oil_map : dict – {commodity_name: yahoo_ticker}
    start   : str  – Start date
    end     : str  – End date

    Returns
    -------
    pd.DataFrame – Combined oil price DataFrame
    """
    all_frames = []

    for commodity_name, ticker_symbol in oil_map.items():
        print(f"  Downloading: {commodity_name} ({ticker_symbol}) … ", end="", flush=True)

        raw_df = download_ticker_data(ticker_symbol, start, end)

        if raw_df is None:
            print("NO DATA")
            continue

        clean_df = clean_and_enrich(raw_df, ticker_symbol)
        clean_df.insert(1, "commodity_name", commodity_name)

        all_frames.append(clean_df)
        print(f"OK  ({len(clean_df):,} rows)")

    if not all_frames:
        raise ValueError("No oil price data was downloaded.")

    combined = pd.concat(all_frames, ignore_index=True)
    print(f"\n  Total oil price rows combined: {len(combined):,}")
    return combined


def save_to_csv(df: pd.DataFrame, filename: str) -> None:
    """
    Save a DataFrame to a CSV file with UTF-8 encoding.
    Prints a confirmation with file size.

    Parameters
    ----------
    df       : pd.DataFrame – Data to save
    filename : str          – Output file path / name
    """
    df.to_csv(filename, index=False, encoding="utf-8-sig")  # utf-8-sig → Excel-friendly BOM

    import os
    size_kb = os.path.getsize(filename) / 1024
    print(f"  Saved: {filename}  ({len(df):,} rows | {size_kb:,.1f} KB)")


def print_summary(df: pd.DataFrame, label: str) -> None:
    """
    Print a quick summary of a DataFrame for quick validation.

    Parameters
    ----------
    df    : pd.DataFrame – DataFrame to summarise
    label : str          – Human-readable label
    """
    print(f"\n{'─'*55}")
    print(f"  SUMMARY: {label}")
    print(f"{'─'*55}")
    print(f"  Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns       : {list(df.columns)}")
    print(f"  Date range    : {df['date'].min()}  →  {df['date'].max()}")

    if "ticker" in df.columns:
        print(f"  Unique tickers: {df['ticker'].nunique()}")
        print(f"  Tickers       : {sorted(df['ticker'].unique().tolist())}")

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f"  Missing values: None")
    else:
        print(f"  Missing values:\n{missing}")

    print(f"{'─'*55}\n")


# ─────────────────────────────────────────────
# 3. MAIN ORCHESTRATION FUNCTION
# ─────────────────────────────────────────────

def main():
    """
    Main entry point – orchestrates data download, processing,
    validation, and persistence.
    """

    # ── Step 1: Download airline stock data ──────────────────
    print("\n[STEP 1] Downloading airline stock data …\n")
    airlines_df = build_airlines_dataframe(
        ticker_map  = AIRLINE_TICKERS,
        start       = START_DATE,
        end         = END_DATE,
        delay_secs  = 0.3,          # 300 ms pause between requests
    )

    # ── Step 2: Download oil price data ──────────────────────
    print("\n[STEP 2] Downloading oil price data …\n")
    oil_df = build_oil_dataframe(
        oil_map = OIL_TICKERS,
        start   = START_DATE,
        end     = END_DATE,
    )
    
# ─────────────────────────────────────────────
# 4. ENTRY POINT
# ─────────────────────────────────────────────

if __name__ == "__main__":
    main()


  AVIATION STOCK & OIL PRICE DATA PIPELINE
  Period : 2015-01-01  →  2026-02-21
  Source : Yahoo Finance (yfinance)

[STEP 1] Downloading airline stock data …

  [01/50] Qatar Airways → SKIPPED (privately held, no public ticker)
  [02/50] Singapore Airlines (C6L.SI) … OK  (2,797 rows)
  [03/50] Cathay Pacific Airways (0293.HK) … OK  (2,742 rows)
  [04/50] Emirates → SKIPPED (privately held, no public ticker)
  [05/50] ANA All Nippon Airways (9202.T) … OK  (2,742 rows)
  [06/50] Turkish Airlines (THYAO.IS) … OK  (2,841 rows)
  [07/50] Korean Air (003490.KS) … OK  (2,729 rows)
  [08/50] Air France-KLM (AF.PA) … OK  (2,852 rows)
  [09/50] Japan Airlines (9201.T) … OK  (2,742 rows)
  [10/50] Hainan Airlines (600221.SS) … OK  (2,702 rows)
  [11/50] Swiss International Air Lines → SKIPPED (privately held, no public ticker)
  [12/50] EVA Air (2618.TW) … OK  (2,705 rows)
  [13/50] British Airways (IAG) (IAG.L) … OK  (2,810 rows)
  [14/50] Qantas Airways (QAN.AX) … OK  (2,820 rows)
  [15/50] Lu

$AIRASIA.KL: possibly delisted; no timezone found

1 Failed download:
['AIRASIA.KL']: possibly delisted; no timezone found


  [WARNING] No data returned for: AIRASIA.KL
NO DATA
  [28/50] Thai Airways (THAI.BK) … OK  (2,707 rows)
  [29/50] Scoot → SKIPPED (privately held, no public ticker)
  [30/50] Fiji Airways → SKIPPED (privately held, no public ticker)
  [31/50] Bangkok Airways (BA.BK) … OK  (2,707 rows)
  [32/50] China Southern Airlines (1055.HK) … OK  (2,742 rows)
  [33/50] Virgin Australia → SKIPPED (privately held, no public ticker)
  [34/50] Gulf Air → SKIPPED (privately held, no public ticker)
  [35/50] Air Astana (AIRA.L) … 

$AIRA.L: possibly delisted; no timezone found

1 Failed download:
['AIRA.L']: possibly delisted; no timezone found


  [WARNING] No data returned for: AIRA.L
NO DATA
  [36/50] China Airlines (2610.TW) … OK  (2,705 rows)
  [37/50] Ethiopian Airlines → SKIPPED (privately held, no public ticker)
  [38/50] IndiGo (INDIGO.NS) … OK  (2,540 rows)
  [39/50] Eurowings → SKIPPED (privately held, no public ticker)
  [40/50] Asiana Airlines (020560.KS) … OK  (2,729 rows)
  [41/50] Vueling Airlines → SKIPPED (privately held, no public ticker)
  [42/50] LATAM Airlines (LTM) … OK  (395 rows)
  [43/50] Porter Airlines → SKIPPED (privately held, no public ticker)
  [44/50] Volotea → SKIPPED (privately held, no public ticker)
  [45/50] Garuda Indonesia (GIAA.JK) … OK  (2,744 rows)
  [46/50] Aegean Airlines (AEGN.AT) … OK  (2,769 rows)
  [47/50] Oman Air → SKIPPED (privately held, no public ticker)
  [48/50] Transavia France → SKIPPED (privately held, no public ticker)
  [49/50] Azerbaijan Airlines → SKIPPED (privately held, no public ticker)
  [50/50] Gulfstream (General Dynamics) (GD) … OK  (2,800 rows)

  Total airl